# I - Introduction

Les déplacements forcés de populations constituent l'un des enjeux géopolitiques et humanitaires majeurs du XXIe siècle. Selon les données du Haut-Commissariat des Nations Unies pour les réfugiés (UNHCR), des millions de personnes sont contraintes chaque année de quitter leur pays d'origine en raison de conflits armés, de persécutions ou d'instabilités politiques. Parmi les grandes régions émettrices mondiales, l'Asie du Sud occupe une place singulière : marquée par des crises prolongées — notamment en Afghanistan, au Pakistan ou encore au Sri Lanka — et par l'exode massif des Rohingyas à partir de 2017, cette région combine à la fois un volume très élevé de demandeurs d'asile et une forte diversité de pays de destination à travers le monde.

Au-delà de l'urgence humanitaire, la répartition géographique de ces flux soulève d'importantes questions économiques et institutionnelles quant aux mécanismes qui gouvernent le choix d'un pays d'accueil. Si les modèles de gravité appliqués aux migrations mettent traditionnellement en avant le rôle central de la proximité géographique et des coûts de transport, les trajectoires des demandeurs d'asile répondent également à des logiques plus complexes : attractivité économique des pays de destination, présence d'un cadre juridique d'asile structuré, proximité culturelle ou linguistique, et liens historiques. L'étude de la période **2017–2020** offre à cet égard un cadre d'observation privilégié : elle permet d'analyser ces déterminants structurels sur quatre années consécutives, avant les bouleversements majeurs induits par les restrictions de mobilité liées à la crise sanitaire puis par les ruptures géopolitiques de 2021.

Ainsi, nous pouvons nous poser la question suivante : **quels sont les facteurs géographiques, économiques et culturels qui déterminent le choix du pays de destination des demandeurs d'asile originaires d'Asie du Sud ?** Plus précisément, la proximité géographique et la richesse du pays d'accueil suffisent-elles à expliquer l'orientation des flux, ou d'autres variables — telles que le partage d'une frontière, d'une langue commune ou de liens coloniaux — jouent-elles un rôle déterminant ?

Ici, nous considérerons que les mouvements migratoires étudiés correspondent aux **nouvelles demandes d'asile** enregistrées annuellement par l'UNHCR entre 2017 et 2020. De plus, nous mesurerons le choix d'une destination par la **part des demandes d'asile** d'un pays d'origine qui s'y dirige (`destination_share`), et nous évaluerons l'influence des facteurs explicatifs en croisant ces flux avec les variables bilatérales de la base *CEPII Gravity* (distance pondérée entre pays, PIB par habitant, contiguïté géographique, langue officielle commune, proximité religieuse et passé colonial).

# II - Données

## Les données UNHCR

L'Agence des Nations Unies pour les réfugiés (UNHCR) met à disposition plusieurs bases de données relatives aux déplacements forcés de population.
Dans le cadre de cette étude, une première source de données est récupérée à partir de l'API de l'UNHCR. L'endpoint consacré aux pays permet notamment d'obtenir les informations suivantes :
- le nom du pays ;
- son code ISO alpha-3, norme internationale permettant d'identifier chaque pays ;
- la région du monde à laquelle il appartient.

Une seconde source de données de l'UNHCR, disponible sous la forme d'un fichier Excel, est utilisée pour mesurer les déplacements entre pays. Elle contient des observations par pays d'origine, pays de destination et année.
Les variables retenues sont :
- le pays d'origine et son code ISO ;
- le pays de destination et son code ISO ;
- l'année ;
- le type de population ou de mouvement (PT) ;
- le nombre d'observations correspondant (Count).

L'objectif de l'étude étant d'analyser la répartition géographique des nouvelles demandes d'asile, seules les observations pour lesquelles PT = ASY sont conservées. Dès lors, la variable Count correspond au nombre de nouvelles demandes d'asile enregistrées pour un pays d'origine, un pays de destination et une année donnés.
Aussi, la variable Count ne doit pas être interprétée comme un nombre d'individus uniques : une même personne peut déposer une demande d'asile dans plusieurs pays au cours de son parcours. L'analyse porte donc sur la distribution des nouvelles demandes d'asile entre pays de destination, et non sur des choix individuels uniques.

In [4]:
# Collecte et manipulation de données
import requests
import numpy as np
import pandas as pd

# Cartographie et visualisation
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
import plotly.express as px

# Modélisation
from sklearn.linear_model import LinearRegression

Les données sont ensuite nettoyées afin d'homogénéiser les types des différentes variables, et seules les observations comprises entre 2017 et 2020 sont conservées.
Enfin, les informations géographiques obtenues via l'API sont jointes aux données relatives aux demandes d'asile grâce aux codes ISO des pays. Cette étape permet notamment d'associer à chaque pays d'origine sa région du monde.
On obtient ainsi le jeu de données `unhcr_df`, qui constitue la base UNHCR utilisée pour la suite de l'analyse.

## Les données CEPII